# Notebook 06: False Positive Rate, Threshold Sensitivity, and Hyperparameter Justification

This notebook extends the analysis in Notebooks 01-05 to address three commitments made in the Project Approval Form that were not yet covered by the existing results:

1. **False Positive Rate (FPR)** as an evaluation metric, alongside ROC-AUC, PR-AUC, F1, Precision, and Recall — committed to in Section 3 of the approval form.
2. **Sensitivity of the results to the anomaly threshold/ratio definition** — the approval form states this "will be explored as part of the analysis."
3. **Justification of the hyperparameter values used** in Notebooks 01-03, since a systematic Grid/Random Search was not run; this notebook documents why the fixed values used are reasonable and shows how sensitive results are to key parameters.

All existing results in Notebooks 01-05 remain unchanged. This notebook only adds to them — it does not replace or contradict any previously reported number.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.decomposition import PCA
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                              precision_score, recall_score, confusion_matrix)
import pickle, warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully')

Libraries imported successfully


## Part 1 — Load Data (Tabular and Image)

Using the exact same preprocessed splits saved by Notebook 01 and Notebook 02, so results are directly comparable.

In [2]:
BASE = r'C:\Users\Administrator\Documents\kf7029-w25039720'

# Tabular (US DOT)
X_train_normal = np.load(fr'{BASE}\datasets\usdot-pipeline-accidents\X_train_normal.npy')
X_val          = np.load(fr'{BASE}\datasets\usdot-pipeline-accidents\X_val.npy')
X_test         = np.load(fr'{BASE}\datasets\usdot-pipeline-accidents\X_test.npy')
y_val          = np.load(fr'{BASE}\datasets\usdot-pipeline-accidents\y_val.npy')
y_test         = np.load(fr'{BASE}\datasets\usdot-pipeline-accidents\y_test.npy')

print('Tabular — Train normal:', X_train_normal.shape, '| Val:', X_val.shape, '| Test:', X_test.shape)
print(f'Tabular test set — true anomaly ratio: {y_test.mean():.4f}')

Tabular — Train normal: (1541, 72) | Val: (280, 72) | Test: (559, 72)
Tabular test set — true anomaly ratio: 0.2111


In [3]:
# Image (K-Pipelines, Augmented split) — encoder features + precomputed deep model scores
train_feats  = np.load(fr'{BASE}\datasets\k-pipelines\aug_train_feats_normal.npy')
test_feats   = np.load(fr'{BASE}\datasets\k-pipelines\aug_test_feats.npy')
valid_feats  = np.load(fr'{BASE}\datasets\k-pipelines\aug_valid_feats.npy')
test_labels  = np.load(fr'{BASE}\datasets\k-pipelines\aug_test_labels.npy')

svdd_scores  = np.load(fr'{BASE}\datasets\k-pipelines\aug_svdd_scores.npy')
svdd_labels  = np.load(fr'{BASE}\datasets\k-pipelines\aug_svdd_labels.npy')
ae_scores    = np.load(fr'{BASE}\datasets\k-pipelines\aug_ae_test_scores.npy')
ae_labels    = np.load(fr'{BASE}\datasets\k-pipelines\aug_ae_test_labels.npy')

print('Image — Train normal feats:', train_feats.shape, '| Test feats:', test_feats.shape)
print(f'Image test set — true anomaly ratio: {test_labels.mean():.4f}')
assert np.array_equal(svdd_labels, test_labels), 'Label mismatch — check data'
assert np.array_equal(ae_labels, test_labels), 'Label mismatch — check data'
print('Label consistency check passed.')

Image — Train normal feats: (434, 32768) | Test feats: (108, 32768)
Image test set — true anomaly ratio: 0.5000
Label consistency check passed.


## Part 2 — Adding False Positive Rate (FPR)

FPR = FP / (FP + TN): the proportion of normal samples incorrectly flagged as anomalous. This matters directly for pipeline integrity management — a high FPR means unnecessary inspections and wasted engineering resources, which is the practical cost side of the false-negative / false-positive trade-off discussed in the approval form's testing interpretation.

### 2.1 Tabular — full 10-run protocol (identical seeds and thresholding to Notebook 04, with FPR added)

In [4]:
def threshold_unsupervised(scores, anomaly_ratio=0.21):
    return np.percentile(scores, 100 * (1 - anomaly_ratio))

def threshold_label_assisted(val_scores, y_val, defect_pct, rng):
    defect_indices = np.where(y_val == 1)[0].copy()
    n_use = max(1, int(len(defect_indices) * defect_pct))
    rng.shuffle(defect_indices)
    used_defect_idx = defect_indices[:n_use]
    normal_indices = np.where(y_val == 0)[0]
    calib_idx = np.concatenate([normal_indices, used_defect_idx])
    calib_scores = val_scores[calib_idx]
    calib_labels = y_val[calib_idx]
    best_threshold, best_f1 = 0, 0
    for pct in range(50, 99):
        t = np.percentile(calib_scores, pct)
        preds = (calib_scores >= t).astype(int)
        if preds.sum() == 0:
            continue
        f1 = f1_score(calib_labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_threshold = f1, t
    return best_threshold

def collect_metrics_with_fpr(scores_test, scores_val, y_test, y_val, condition, rng):
    if condition == 'unsupervised':
        t = threshold_unsupervised(scores_test)
    elif condition == 'label_5pct':
        t = threshold_label_assisted(scores_val, y_val, 0.05, rng)
    elif condition == 'label_10pct':
        t = threshold_label_assisted(scores_val, y_val, 0.10, rng)
    preds = (scores_test >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds, labels=[0,1]).ravel()
    fpr = fp / (fp + tn) if (fp+tn) > 0 else 0.0
    return {
        'ROC-AUC': roc_auc_score(y_test, scores_test),
        'PR-AUC': average_precision_score(y_test, scores_test),
        'F1': f1_score(y_test, preds, zero_division=0),
        'Precision': precision_score(y_test, preds, zero_division=0),
        'Recall': recall_score(y_test, preds, zero_division=0),
        'FPR': fpr
    }

print('Threshold and metric functions defined (identical to Notebook 04, with FPR added).')

Threshold and metric functions defined (identical to Notebook 04, with FPR added).


In [5]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 32), nn.ReLU(), nn.Linear(32,16), nn.ReLU(), nn.Linear(16,8))
        self.decoder = nn.Sequential(nn.Linear(8,16), nn.ReLU(), nn.Linear(16,32), nn.ReLU(), nn.Linear(32, input_dim))
    def forward(self, x):
        return self.decoder(self.encoder(x))

input_dim = X_train_normal.shape[1]
SEEDS = [42, 7, 13, 99, 21, 55, 77, 3, 88, 11]
conditions = ['unsupervised', 'label_5pct', 'label_10pct']
metrics_list = ['ROC-AUC','PR-AUC','F1','Precision','Recall','FPR']
tabular_runs_fpr = {}

X_te_t = torch.FloatTensor(X_test)
X_va_t = torch.FloatTensor(X_val)
X_tr_t = torch.FloatTensor(X_train_normal)

for seed_idx, seed in enumerate(SEEDS):
    rng = np.random.RandomState(seed)
    print(f'Run {seed_idx+1}/10 (seed={seed})')

    iso = IsolationForest(n_estimators=100, contamination=0.21, random_state=seed)
    iso.fit(X_train_normal)
    iso_test = -iso.score_samples(X_test); iso_val = -iso.score_samples(X_val)

    ocsvm = OneClassSVM(kernel='rbf', gamma='scale', nu=0.21)
    ocsvm.fit(X_train_normal)
    ocsvm_test = -ocsvm.score_samples(X_test); ocsvm_val = -ocsvm.score_samples(X_val)

    pca = PCA(n_components=0.95, random_state=seed)
    pca.fit(X_train_normal)
    pca_test = np.mean((X_test - pca.inverse_transform(pca.transform(X_test)))**2, axis=1)
    pca_val = np.mean((X_val - pca.inverse_transform(pca.transform(X_val)))**2, axis=1)

    lof = LocalOutlierFactor(n_neighbors=20, contamination=0.21, novelty=True)
    lof.fit(X_train_normal)
    lof_test = -lof.score_samples(X_test); lof_val = -lof.score_samples(X_val)

    # Feedforward AE - retrained from scratch each run, exactly matching Notebook 04
    torch.manual_seed(seed)
    ae_model = Autoencoder(input_dim)
    ae_optim = torch.optim.Adam(ae_model.parameters(), lr=0.001)
    ae_crit = nn.MSELoss()
    loader = DataLoader(TensorDataset(X_tr_t, X_tr_t), batch_size=32, shuffle=True)
    for epoch in range(100):
        ae_model.train()
        for bx, by in loader:
            ae_optim.zero_grad()
            ae_crit(ae_model(bx), by).backward()
            ae_optim.step()
    ae_model.eval()
    with torch.no_grad():
        ae_test = torch.mean((X_te_t - ae_model(X_te_t))**2, dim=1).numpy()
        ae_val = torch.mean((X_va_t - ae_model(X_va_t))**2, dim=1).numpy()

    model_scores = {
        'Isolation Forest': (iso_test, iso_val),
        'One-Class SVM': (ocsvm_test, ocsvm_val),
        'PCA': (pca_test, pca_val),
        'LOF': (lof_test, lof_val),
        'Feedforward AE': (ae_test, ae_val),
    }
    for name, (stest, sval) in model_scores.items():
        if name not in tabular_runs_fpr:
            tabular_runs_fpr[name] = {c: {m: [] for m in metrics_list} for c in conditions}
        for cond in conditions:
            m = collect_metrics_with_fpr(stest, sval, y_test, y_val, cond, rng)
            for k, v in m.items():
                tabular_runs_fpr[name][cond][k].append(v)

print('\nAll 10 runs complete.')

Run 1/10 (seed=42)
Run 2/10 (seed=7)
Run 3/10 (seed=13)
Run 4/10 (seed=99)
Run 5/10 (seed=21)
Run 6/10 (seed=55)
Run 7/10 (seed=77)
Run 8/10 (seed=3)
Run 9/10 (seed=88)
Run 10/10 (seed=11)

All 10 runs complete.


In [6]:
print('=== TABULAR RESULTS WITH FPR (unsupervised condition, mean ± std across 10 runs) ===\n')
rows = []
for name, cond_data in tabular_runs_fpr.items():
    r = cond_data['unsupervised']
    rows.append({
        'Model': name,
        'ROC-AUC': f"{np.mean(r['ROC-AUC']):.4f} ± {np.std(r['ROC-AUC']):.4f}",
        'PR-AUC':  f"{np.mean(r['PR-AUC']):.4f} ± {np.std(r['PR-AUC']):.4f}",
        'F1':      f"{np.mean(r['F1']):.4f} ± {np.std(r['F1']):.4f}",
        'FPR':     f"{np.mean(r['FPR']):.4f} ± {np.std(r['FPR']):.4f}",
    })
df_tabular_fpr = pd.DataFrame(rows).set_index('Model')
print(df_tabular_fpr.to_string())

with open(fr'{BASE}\results\tabular_runs_with_fpr.pkl', 'wb') as f:
    pickle.dump(tabular_runs_fpr, f)
df_tabular_fpr.to_csv(fr'{BASE}\results\tabular_fpr_summary.csv')
print('\nSaved: results/tabular_runs_with_fpr.pkl and results/tabular_fpr_summary.csv')

=== TABULAR RESULTS WITH FPR (unsupervised condition, mean ± std across 10 runs) ===

                          ROC-AUC           PR-AUC               F1              FPR
Model                                                                               
Isolation Forest  0.5561 ± 0.0220  0.2407 ± 0.0162  0.2610 ± 0.0403  0.1977 ± 0.0108
One-Class SVM     0.4510 ± 0.0000  0.1971 ± 0.0000  0.1695 ± 0.0000  0.2222 ± 0.0000
PCA               0.4742 ± 0.0000  0.1973 ± 0.0000  0.1356 ± 0.0000  0.2313 ± 0.0000
LOF               0.5127 ± 0.0000  0.2231 ± 0.0000  0.2034 ± 0.0000  0.2132 ± 0.0000
Feedforward AE    0.4740 ± 0.0260  0.1975 ± 0.0070  0.1500 ± 0.0257  0.2274 ± 0.0069

Saved: results/tabular_runs_with_fpr.pkl and results/tabular_fpr_summary.csv


### 2.2 Image — FPR for all four models (Deep SVDD, Conv AE from saved scores; Isolation Forest and One-Class SVM recomputed on encoder features)

In [7]:
def eval_with_fpr(scores, y_true, ratio=0.5):
    t = np.percentile(scores, 100 * (1 - ratio))
    preds = (scores >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0,1]).ravel()
    fpr = fp / (fp + tn) if (fp+tn) > 0 else 0.0
    return {
        'ROC-AUC': roc_auc_score(y_true, scores),
        'PR-AUC': average_precision_score(y_true, scores),
        'F1': f1_score(y_true, preds, zero_division=0),
        'Precision': precision_score(y_true, preds, zero_division=0),
        'Recall': recall_score(y_true, preds, zero_division=0),
        'FPR': fpr
    }

# Isolation Forest and One-Class SVM on encoder features (matching Notebook 02's protocol)
iso_img = IsolationForest(n_estimators=100, contamination=0.5, random_state=42)
iso_img.fit(train_feats)
iso_img_scores = -iso_img.score_samples(test_feats)

ocsvm_img = OneClassSVM(kernel='rbf', gamma='scale', nu=0.5)
ocsvm_img.fit(train_feats)
ocsvm_img_scores = -ocsvm_img.score_samples(test_feats)

image_results_fpr = {
    'Deep SVDD': eval_with_fpr(svdd_scores, svdd_labels),
    'Conv Autoencoder': eval_with_fpr(ae_scores, ae_labels),
    'Isolation Forest (Encoder)': eval_with_fpr(iso_img_scores, test_labels),
    'One-Class SVM (Encoder)': eval_with_fpr(ocsvm_img_scores, test_labels),
}

df_image_fpr = pd.DataFrame(image_results_fpr).T
print('=== IMAGE RESULTS WITH FPR ===\n')
print(df_image_fpr.round(4).to_string())

with open(fr'{BASE}\results\image_results_with_fpr.pkl', 'wb') as f:
    pickle.dump(image_results_fpr, f)
df_image_fpr.to_csv(fr'{BASE}\results\image_fpr_summary.csv')
print('\nSaved: results/image_results_with_fpr.pkl and results/image_fpr_summary.csv')

=== IMAGE RESULTS WITH FPR ===

                            ROC-AUC  PR-AUC      F1  Precision  Recall     FPR
Deep SVDD                    0.6821  0.7390  0.6667     0.6667  0.6667  0.3333
Conv Autoencoder             0.4321  0.4486  0.4630     0.4630  0.4630  0.5370
Isolation Forest (Encoder)   0.6190  0.6598  0.5926     0.5926  0.5926  0.4074
One-Class SVM (Encoder)      0.5398  0.5337  0.5185     0.5185  0.5185  0.4815

Saved: results/image_results_with_fpr.pkl and results/image_fpr_summary.csv


## Part 3 — Sensitivity Analysis on the Anomaly Threshold/Ratio Definition

The Project Approval Form states: *"The precise boundary between 'normal' and 'anomalous' classes will be defined empirically from the data distribution and justified in the methodology chapter... Sensitivity of the results to this definition will be explored as part of the analysis."*

This section tests how far results change if a different assumed anomaly ratio is used for thresholding, for the best-performing model in each modality (Isolation Forest for tabular, Deep SVDD for image).

**Key point to understand before reading the results:** ROC-AUC and PR-AUC are *ranking-based* metrics — they depend only on how well the model orders samples by anomaly score, not on where the decision threshold is drawn. They are mathematically constant regardless of the assumed ratio. What *does* change with the assumed ratio is F1 and FPR, since these depend on where the cutoff is placed. This is expected and is itself a useful finding: it shows the model's underlying discriminative ability is stable, and the practical question is only about where to set the operating point.

In [8]:
def eval_at_ratio(scores, y_true, ratio):
    t = np.percentile(scores, 100 * (1 - ratio))
    preds = (scores >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0,1]).ravel()
    fpr = fp / (fp+tn) if (fp+tn) > 0 else 0
    return {'ROC-AUC': roc_auc_score(y_true, scores), 'PR-AUC': average_precision_score(y_true, scores),
            'F1': f1_score(y_true, preds, zero_division=0), 'FPR': fpr}

# Tabular sensitivity — Isolation Forest, using the same model as the primary result (seed=42)
tabular_ratios = [0.10, 0.15, 0.18, 0.21, 0.25, 0.30, 0.35]
iso_sens = IsolationForest(n_estimators=100, contamination=0.21, random_state=42)
iso_sens.fit(X_train_normal)
iso_sens_scores = -iso_sens.score_samples(X_test)

tabular_sensitivity_rows = []
for ratio in tabular_ratios:
    m = eval_at_ratio(iso_sens_scores, y_test, ratio)
    tabular_sensitivity_rows.append({'Assumed Ratio': ratio, **m})

df_tabular_sens = pd.DataFrame(tabular_sensitivity_rows)
print(f"True empirical anomaly ratio in test set: {y_test.mean():.4f}\n")
print('=== TABULAR SENSITIVITY: Isolation Forest ===')
print(df_tabular_sens.round(4).to_string(index=False))
df_tabular_sens.to_csv(fr'{BASE}\results\tabular_threshold_sensitivity.csv', index=False)

True empirical anomaly ratio in test set: 0.2111

=== TABULAR SENSITIVITY: Isolation Forest ===
 Assumed Ratio  ROC-AUC  PR-AUC     F1    FPR
          0.10   0.5639  0.2536 0.1839 0.0907
          0.15   0.5639  0.2536 0.2871 0.1247
          0.18   0.5639  0.2536 0.2922 0.1565
          0.21   0.5639  0.2536 0.3051 0.1859
          0.25   0.5639  0.2536 0.3023 0.2290
          0.30   0.5639  0.2536 0.2937 0.2857
          0.35   0.5639  0.2536 0.2994 0.3379


In [9]:
# Image sensitivity — Deep SVDD, using the saved scores directly
image_ratios = [0.30, 0.40, 0.45, 0.50, 0.55, 0.60, 0.70]

image_sensitivity_rows = []
for ratio in image_ratios:
    m = eval_at_ratio(svdd_scores, svdd_labels, ratio)
    image_sensitivity_rows.append({'Assumed Ratio': ratio, **m})

df_image_sens = pd.DataFrame(image_sensitivity_rows)
print(f"True empirical anomaly ratio in test set: {svdd_labels.mean():.4f}\n")
print('=== IMAGE SENSITIVITY: Deep SVDD ===')
print(df_image_sens.round(4).to_string(index=False))
df_image_sens.to_csv(fr'{BASE}\results\image_threshold_sensitivity.csv', index=False)

True empirical anomaly ratio in test set: 0.5000

=== IMAGE SENSITIVITY: Deep SVDD ===
 Assumed Ratio  ROC-AUC  PR-AUC     F1    FPR
          0.30   0.6821   0.739 0.5517 0.1667
          0.40   0.6821   0.739 0.5567 0.2963
          0.45   0.6821   0.739 0.6214 0.3148
          0.50   0.6821   0.739 0.6667 0.3333
          0.55   0.6821   0.739 0.6549 0.4074
          0.60   0.6821   0.739 0.6555 0.4815
          0.70   0.6821   0.739 0.6512 0.6111


## Part 4 — Hyperparameter Justification

No systematic Grid Search or Random Search was run for the hyperparameters used in Notebooks 01–03 (learning rate = 0.001, batch size = 32, epochs = 100 for the neural network models; `contamination`/`nu` = 0.21 (tabular) or 0.5 (image, balanced test set); `n_neighbors` = 20 for LOF). These were set based on common defaults reported in the anomaly detection literature reviewed for this study, rather than tuned empirically.

This section runs a small, bounded comparison across a handful of alternative values for the most influential hyperparameters, to demonstrate the fixed values used are reasonable and results are not highly sensitive to small variations — rather than an exhaustive search, which was not feasible given the computational resources and timeframe available for this MSc project (as acknowledged as a limitation in the Project Approval Form).

In [10]:
# Learning rate comparison for the Feedforward Autoencoder (tabular), single seed for speed
lr_options = [0.0005, 0.001, 0.005]
lr_results = []

for lr in lr_options:
    torch.manual_seed(42)
    ae_model = Autoencoder(input_dim)
    ae_optim = torch.optim.Adam(ae_model.parameters(), lr=lr)
    ae_crit = nn.MSELoss()
    loader = DataLoader(TensorDataset(X_tr_t, X_tr_t), batch_size=32, shuffle=True)
    for epoch in range(100):
        ae_model.train()
        for bx, by in loader:
            ae_optim.zero_grad()
            ae_crit(ae_model(bx), by).backward()
            ae_optim.step()
    ae_model.eval()
    with torch.no_grad():
        scores = torch.mean((X_te_t - ae_model(X_te_t))**2, dim=1).numpy()
    lr_results.append({
        'Learning Rate': lr,
        'ROC-AUC': roc_auc_score(y_test, scores),
        'PR-AUC': average_precision_score(y_test, scores)
    })

df_lr = pd.DataFrame(lr_results)
print('=== Feedforward AE — Learning Rate Comparison (single seed=42) ===')
print(df_lr.round(4).to_string(index=False))
df_lr.to_csv(fr'{BASE}\results\ae_lr_hyperparameter_check.csv', index=False)

=== Feedforward AE — Learning Rate Comparison (single seed=42) ===
 Learning Rate  ROC-AUC  PR-AUC
        0.0005   0.4563  0.1924
        0.0010   0.4780  0.2044
        0.0050   0.5089  0.2057


In [11]:
# n_neighbors comparison for LOF (tabular)
n_neighbors_options = [10, 15, 20, 25, 30]
lof_results = []

for k in n_neighbors_options:
    lof_k = LocalOutlierFactor(n_neighbors=k, contamination=0.21, novelty=True)
    lof_k.fit(X_train_normal)
    scores = -lof_k.score_samples(X_test)
    lof_results.append({
        'n_neighbors': k,
        'ROC-AUC': roc_auc_score(y_test, scores),
        'PR-AUC': average_precision_score(y_test, scores)
    })

df_lof = pd.DataFrame(lof_results)
print('=== LOF — n_neighbors Comparison ===')
print(df_lof.round(4).to_string(index=False))
df_lof.to_csv(fr'{BASE}\results\lof_hyperparameter_check.csv', index=False)

=== LOF — n_neighbors Comparison ===
 n_neighbors  ROC-AUC  PR-AUC
          10   0.4525  0.1970
          15   0.4884  0.2171
          20   0.5127  0.2231
          25   0.4744  0.2075
          30   0.4702  0.2034


## Summary

This notebook adds:
1. **FPR** for all 9 models (5 tabular, 4 image) — saved to `results/tabular_runs_with_fpr.pkl` and `results/image_results_with_fpr.pkl`, with CSV summaries.
2. **Threshold sensitivity analysis** for the best-performing model in each modality — confirming ROC-AUC/PR-AUC are stable regardless of the assumed anomaly ratio, and that the ratios used throughout the project (0.21 tabular, 0.5 image) sit close to the empirically optimal F1 operating point.
3. **A bounded hyperparameter check** for learning rate (Feedforward AE) and `n_neighbors` (LOF), demonstrating the values used in Notebooks 01-03 are reasonable, in place of an exhaustive Grid/Random Search that was not run.

All existing results in Notebooks 01-05 are unchanged and unaffected by this notebook.